# OCR – Savings Book Transaction Page

**Pipeline:** Load → Preprocess → OCR per ROI → Parse & Structure → Confidence & Save

Mỗi cell in kết quả trung gian của từng bước để dễ debug.

In [ ]:
from pathlib import Path
import re
from dataclasses import dataclass

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from paddleocr import PaddleOCR

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "extract":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

SAVINGS_ROOT = PROJECT_ROOT / "data" / "unstructured" / "documents" / "doc_type=savings_book"
run_dirs = sorted(SAVINGS_ROOT.glob("run_date=*"))
INPUT_DIR = run_dirs[-1] if run_dirs else SAVINGS_ROOT / "run_date=2026-05-25"
RUN_DATE = INPUT_DIR.name.split("=", 1)[1]

_n = input("Số lượng ảnh cần xử lý (Enter = tất cả): ").strip()
LIMIT = int(_n) if _n else 0
PREVIEW_N = min(3, LIMIT) if LIMIT else 3
OCR_LANG = "en"

img_paths = sorted(p for ext in ("*.jpg", "*.jpeg", "*.png", "*.tif", "*.tiff")
                   for p in INPUT_DIR.rglob(ext))
if LIMIT:
    img_paths = img_paths[:LIMIT]

print(f"INPUT_DIR : {INPUT_DIR}")
print(f"RUN_DATE  : {RUN_DATE}")
print(f"Images    : {len(img_paths)}  (limit={LIMIT or 'all'})")
print(f"OCR_LANG  : {OCR_LANG}")
pd.DataFrame({"image_path": [str(p) for p in img_paths]}).head(20)


In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def show_bgr(title: str, bgr: np.ndarray, max_w: int = 8) -> None:
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if bgr.ndim == 3 else bgr
    h, w = rgb.shape[:2]
    plt.figure(figsize=(max_w, max_w * h / max(1, w)))
    plt.imshow(rgb, cmap="gray" if rgb.ndim == 2 else None)
    plt.title(title); plt.axis("off"); plt.show()

def normalize_width(bgr: np.ndarray, out_w: int = 1200) -> np.ndarray:
    h, w = bgr.shape[:2]
    if w == out_w:
        return bgr
    out_h = int(round(h * out_w / max(1, w)))
    interp = cv2.INTER_AREA if out_w < w else cv2.INTER_CUBIC
    return cv2.resize(bgr, (out_w, out_h), interpolation=interp)

def rotate_bound(bgr: np.ndarray, angle: float, border=(245, 245, 245)) -> np.ndarray:
    h, w = bgr.shape[:2]
    cx, cy = w / 2.0, h / 2.0
    m = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    cos, sin = abs(m[0, 0]), abs(m[0, 1])
    new_w, new_h = int(h * sin + w * cos), int(h * cos + w * sin)
    m[0, 2] += new_w / 2.0 - cx
    m[1, 2] += new_h / 2.0 - cy
    return cv2.warpAffine(bgr, m, (new_w, new_h), flags=cv2.INTER_CUBIC, borderValue=border)

def estimate_skew_angle(bgr: np.ndarray) -> float:
    gray = cv2.GaussianBlur(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (3, 3), 0)
    lines = cv2.HoughLinesP(
        cv2.Canny(gray, 50, 150, apertureSize=3), 1, np.pi / 180,
        threshold=160, minLineLength=max(120, bgr.shape[1] // 5), maxLineGap=20,
    )
    if lines is None:
        return 0.0
    angles = [np.degrees(np.arctan2(y2 - y1, x2 - x1))
              for x1, y1, x2, y2 in lines[:, 0, :]
              if -15 <= np.degrees(np.arctan2(y2 - y1, x2 - x1)) <= 15]
    return float(np.median(angles)) if angles else 0.0

def deskew(bgr: np.ndarray, min_abs: float = 0.20) -> tuple[np.ndarray, float]:
    angle = estimate_skew_angle(bgr)
    if abs(angle) < min_abs:
        return bgr, angle
    return rotate_bound(bgr, angle=-angle), angle

def order_points(pts: np.ndarray) -> np.ndarray:
    rect = np.zeros((4, 2), dtype="float32")
    s, diff = pts.sum(axis=1), np.diff(pts, axis=1)
    rect[0], rect[2] = pts[np.argmin(s)], pts[np.argmax(s)]
    rect[1], rect[3] = pts[np.argmin(diff)], pts[np.argmax(diff)]
    return rect

def find_document_quad(bgr: np.ndarray) -> np.ndarray | None:
    blur = cv2.GaussianBlur(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    edges = cv2.Canny(blur, 40, 120)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    img_area = bgr.shape[0] * bgr.shape[1]
    for c in sorted(contours, key=cv2.contourArea, reverse=True)[:10]:
        if cv2.contourArea(c) < img_area * 0.25:
            continue
        approx = cv2.approxPolyDP(c, 0.02 * cv2.arcLength(c, True), True)
        if len(approx) == 4:
            return order_points(approx.reshape(4, 2).astype("float32"))
    return None

def warp_document(bgr: np.ndarray, quad: np.ndarray, out_w: int = 1200) -> np.ndarray:
    rect = order_points(quad)
    tl, tr, br, bl = rect
    max_w = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    max_h = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    dst = np.array([[0, 0], [max_w - 1, 0], [max_w - 1, max_h - 1], [0, max_h - 1]], dtype="float32")
    warped = cv2.warpPerspective(bgr, cv2.getPerspectiveTransform(rect, dst),
                                 (max_w, max_h), borderValue=(245, 245, 245))
    return normalize_width(warped, out_w=out_w)

def preprocess_for_ocr(bgr: np.ndarray) -> dict[str, np.ndarray]:
    gray    = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    denoise = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(denoise)
    sharpen = cv2.addWeighted(clahe, 1.5, cv2.GaussianBlur(clahe, (0, 0), 1.0), -0.5, 0)
    return {"bgr": bgr, "gray": gray, "clahe": clahe, "sharpen": sharpen}


# ── ROI definitions ───────────────────────────────────────────────────────────
# Fractions relative to the WARPED image (warp_document crops to page border).
# Source image: 1654×2339 px, margin=80 → page border w=1494, h=2179.
# After warp to out_w=1200: scale=0.803, warped height≈1749 px.
# col_x=[90,330,450,930,1210,1360,1564] → relative to border → warped fractions.
# row1_top=270 → date y≈0.095, description y≈0.119 in warped image.

@dataclass
class ROI:
    name: str
    x1: float; y1: float; x2: float; y2: float

SAVINGS_BOOK_ROIS = [
    ROI("transaction_date",   0.007, 0.088, 0.167, 0.115),
    ROI("description",        0.007, 0.115, 0.167, 0.142),
    ROI("transaction_code",   0.167, 0.088, 0.248, 0.142),
    ROI("transaction_amount", 0.248, 0.088, 0.569, 0.115),
    ROI("balance",            0.569, 0.088, 0.757, 0.115),
    ROI("interest_rate",      0.757, 0.088, 0.857, 0.115),
    ROI("signature",          0.857, 0.108, 0.993, 0.162),
]

def scale_rois(rois: list[ROI], w: int, h: int) -> dict[str, tuple[int, int, int, int]]:
    return {r.name: (
        max(0, min(w - 1, int(round(r.x1 * w)))),
        max(0, min(h - 1, int(round(r.y1 * h)))),
        max(0, min(w,     int(round(r.x2 * w)))),
        max(0, min(h,     int(round(r.y2 * h)))),
    ) for r in rois}


# ── Text helpers ──────────────────────────────────────────────────────────────

_WATERMARK_PATTERNS = [
    r"\bSAMPLE\b",
    r"\bDEMO\s*-?\s*NOT\s+A\s+REAL\s+BANK\s+DOCUMENT\b",
    r"\bNOT\s+A\s+REAL\s+BANK\s+DOCUMENT\b",
    r"\bFOR\s+TESTING\s+ONLY\b",
    r"\bDEMO\s+BANK\b",
]

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "").replace("|", "I")).strip()

def strip_watermark(text: str) -> str:
    for p in _WATERMARK_PATTERNS:
        text = re.sub(p, " ", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()

def parse_field(field: str, text: str) -> str:
    text = strip_watermark(clean_text(text))
    if field == "transaction_date":
        m = re.search(r"\d{1,2}[/\-.]\d{1,2}[/\-.]\d{2,4}", text)
        return m.group(0).replace("-", "/").replace(".", "/") if m else text
    if field == "transaction_code":
        m = re.search(r"\b[A-Z]{2,8}\b", text.upper())
        return m.group(0) if m else text.upper()
    if field in {"transaction_amount", "balance"}:
        m = re.search(r"\d[\d,\.]*", text)
        return m.group(0) if m else text
    if field == "interest_rate":
        m = re.search(r"\d+(?:[\.,]\d+)?\s*%", text)
        return m.group(0).replace(",", ".") if m else text
    if field == "signature":
        text = re.sub(r"\b(DEMO|Signature)\b", " ", text, flags=re.IGNORECASE)
        return clean_text(text)
    return text

def is_plausible(field: str, text: str) -> bool:
    text = clean_text(text)
    if not text:
        return False
    if field == "transaction_date":
        return bool(re.search(r"\d{1,2}/\d{1,2}/\d{2,4}", text))
    if field == "transaction_code":
        return bool(re.fullmatch(r"[A-Z]{2,8}", text))
    if field in {"transaction_amount", "balance"}:
        return bool(re.search(r"\d", text))
    if field == "interest_rate":
        return "%" in text and bool(re.search(r"\d", text))
    return len(text) >= 2

def extract_user_id(path: Path) -> int | None:
    for part in path.parts:
        if part.startswith("user_id="):
            try: return int(part.split("=", 1)[1])
            except Exception: return None
    return None

print("Helpers loaded — ROI fields:", [r.name for r in SAVINGS_BOOK_ROIS])


## Step 1 – Load Images
Load ảnh đầu vào và in thông tin cơ bản của từng file.

In [ ]:
# Step 1: Load Images
input_images = []
input_rows = []

for path in img_paths:
    bgr = cv2.imread(str(path))
    status = "ok" if bgr is not None else "read_error"
    row = {
        "file": path.name,
        "file_path": str(path),
        "user_id": extract_user_id(path),
        "status": status,
    }
    if bgr is not None:
        row.update({"height": bgr.shape[0], "width": bgr.shape[1]})
        input_images.append({"path": path, "bgr": bgr, **row})
    input_rows.append(row)

print("=== STEP 1: LOAD IMAGES ===")
print(f"Loaded {len(input_images)}/{len(img_paths)} images")
df_input = pd.DataFrame(input_rows)
display(df_input)

for item in input_images[:PREVIEW_N]:
    show_bgr(f"Input – user_id={item['user_id']} – {item['file']}", item["bgr"], max_w=7)

## Step 2 – Preprocess
Deskew, detect/crop trang tài liệu, warp về chiều rộng chuẩn, tạo các biến thể gray/CLAHE/sharpen.

In [ ]:
# Step 2: Preprocess
preprocessed_docs = []
preprocess_rows = []

for item in input_images:
    bgr = item["bgr"]
    deskewed, skew_angle = deskew(bgr)
    quad = find_document_quad(deskewed)
    warped = warp_document(deskewed, quad, out_w=1200) if quad is not None else normalize_width(deskewed, out_w=1200)
    pp = preprocess_for_ocr(warped)
    roi_boxes = scale_rois(SAVINGS_BOOK_ROIS, warped.shape[1], warped.shape[0])

    preprocessed_docs.append({**item, "deskewed": deskewed, "warped": warped, "pp": pp, "roi_boxes": roi_boxes})
    preprocess_rows.append({
        "file": item["file"],
        "user_id": item["user_id"],
        "skew_angle": round(skew_angle, 3),
        "quad_found": quad is not None,
        "warped_size": f"{warped.shape[1]}×{warped.shape[0]}",
        "roi_count": len(roi_boxes),
    })

print("=== STEP 2: PREPROCESS ===")
df_preprocess = pd.DataFrame(preprocess_rows)
display(df_preprocess)

for item in preprocessed_docs[:PREVIEW_N]:
    show_bgr(f"Warped – user_id={item['user_id']}", item["warped"], max_w=7)

In [ ]:
# DEBUG: Vẽ ROI boxes lên ảnh warped để kiểm tra tọa độ có đúng không
# Chạy sau Step 2 để xác nhận ROI căn chỉnh đúng trước khi OCR

COLORS = {
    "transaction_date":   (0,   200,  50),
    "description":        (0,   100, 255),
    "transaction_code":   (200,  50, 200),
    "transaction_amount": (255, 140,   0),
    "balance":            (0,   200, 200),
    "interest_rate":      (200, 200,   0),
    "signature":          (200,   0,   0),
}

def draw_roi_boxes(bgr: np.ndarray, roi_boxes: dict, scale: float = 0.5) -> np.ndarray:
    out = bgr.copy()
    h, w = out.shape[:2]
    font = cv2.FONT_HERSHEY_SIMPLEX
    for name, (x1, y1, x2, y2) in roi_boxes.items():
        color = COLORS.get(name, (128, 128, 128))
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 3)
        cv2.putText(out, name, (x1 + 4, max(y1 - 6, 15)), font, 0.55, color, 2, cv2.LINE_AA)
    if scale != 1.0:
        out = cv2.resize(out, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return out

print("=== DEBUG: ROI VISUALIZATION ===")
for item in preprocessed_docs[:PREVIEW_N]:
    vis = draw_roi_boxes(item["warped"], item["roi_boxes"], scale=0.5)
    show_bgr(
        f"ROI boxes – user_id={item['user_id']} | warped {item['warped'].shape[1]}×{item['warped'].shape[0]}",
        vis, max_w=10,
    )
    H, W = item["warped"].shape[:2]
    print(f"user_id={item['user_id']} — pixel ROI boxes (x1,y1,x2,y2):")
    for name, (x1, y1, x2, y2) in item["roi_boxes"].items():
        print(f"  {name:22s}: px=({x1},{y1},{x2},{y2})  frac=(x:{x1/W:.3f}–{x2/W:.3f}, y:{y1/H:.3f}–{y2/H:.3f})")

## Step 3 – OCR per ROI
Khởi tạo PaddleOCR và nhận dạng văn bản từng ROI ở nhiều biến thể ảnh; chọn kết quả tốt nhất theo plausibility + score.

In [ ]:
# Step 3: OCR per ROI
ocr = PaddleOCR(lang=OCR_LANG, ocr_version="PP-OCRv3", use_textline_orientation=False)

def _box_to_xyxy(box) -> tuple | None:
    arr = np.asarray(box, dtype=np.float32)
    if arr.size == 4 and arr.ndim == 1:
        return tuple(arr.tolist())
    if arr.ndim >= 2 and arr.shape[-1] >= 2:
        pts = arr.reshape(-1, arr.shape[-1])[:, :2]
        return float(np.min(pts[:, 0])), float(np.min(pts[:, 1])), float(np.max(pts[:, 0])), float(np.max(pts[:, 1]))
    return None

def ocr_predict_items(img: np.ndarray) -> tuple[list[dict], str, float | None]:
    bgr_img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR) if img.ndim == 2 else img
    results = ocr.predict(
        bgr_img,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )
    if not results:
        return [], "", None
    j = getattr(results[0], "json", None)
    if not isinstance(j, dict):
        return [], "", None
    res    = j.get("res", {})
    texts  = res.get("rec_texts",  []) or []
    scores = res.get("rec_scores", []) or []
    boxes  = next((res.get(k) for k in ("rec_boxes", "rec_polys", "dt_boxes", "dt_polys") if res.get(k)), [])
    items = []
    for i, text in enumerate(texts):
        if i >= len(boxes):
            continue
        xyxy = _box_to_xyxy(boxes[i])
        if xyxy is None:
            continue
        score = float(scores[i] if i < len(scores) and scores[i] is not None else 0.0)
        x1, y1, x2, y2 = xyxy
        items.append({"text": str(text).strip(), "score": score, "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    full_text  = " ".join(it["text"] for it in items).strip()
    mean_score = float(np.mean([it["score"] for it in items])) if items else None
    return items, full_text, mean_score

def ocr_roi_variants(pp: dict, box: tuple) -> list[tuple[str, np.ndarray]]:
    x1, y1, x2, y2 = box
    variants = []
    for key in ("sharpen", "clahe", "gray"):
        base = pp.get(key)
        if base is None:
            continue
        roi = base[y1:y2, x1:x2]
        if roi.size:
            variants.append((key, roi))
            variants.append((f"{key}_x2", cv2.resize(roi, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)))
    return variants

def choose_best_ocr(field_name: str, candidates: list[tuple]) -> tuple:
    best, best_key = ("", None, "", 0), (-1, -1.0, -1, -1)
    for src, raw, score, n_items in candidates:
        parsed = parse_field(field_name, raw)
        plaus  = 1 if is_plausible(field_name, parsed) else 0
        s      = float(score) if score is not None else -1.0
        key    = (plaus, s, len(parsed), n_items)
        if key > best_key:
            best_key = key
            best     = (raw, score, src, n_items)
    return best

recognized_docs = []
recognition_rows = []

for item in preprocessed_docs:
    field_raw = {}
    for field_name, box in item["roi_boxes"].items():
        candidates = []
        for src, roi in ocr_roi_variants(item["pp"], box):
            items_list, full_text, mean_score = ocr_predict_items(roi)
            candidates.append((src, full_text, mean_score, len(items_list)))
        raw, score, src, _ = choose_best_ocr(field_name, candidates)
        field_raw[field_name] = {"raw_text": raw, "score": score, "src": src}
        recognition_rows.append({
            "file": item["file"],
            "user_id": item["user_id"],
            "field": field_name,
            "raw_text": raw,
            "score": round(score, 4) if score is not None else None,
            "variant": src,
        })
    recognized_docs.append({**item, "field_raw": field_raw})

print("=== STEP 3: OCR PER ROI ===")
df_recognition = pd.DataFrame(recognition_rows)
display(df_recognition)

for item in recognized_docs[:PREVIEW_N]:
    print(f"\nROI preview – user_id={item['user_id']} – {item['file']}")
    for field_name, box in item["roi_boxes"].items():
        x1, y1, x2, y2 = box
        roi = item["warped"][y1:y2, x1:x2]
        show_bgr(f"ROI {field_name}: {item['field_raw'][field_name]['raw_text']}", roi, max_w=5)

## Step 4 – Parse & Structure
Làm sạch watermark, chuẩn hóa từng field và gom thành record có cấu trúc.

In [ ]:
# Step 4: Parse & Structure
structured_docs = []
structured_rows = []

for item in recognized_docs:
    parsed_fields = {
        field_name: parse_field(field_name, payload["raw_text"])
        for field_name, payload in item["field_raw"].items()
    }
    structured_docs.append({**item, "parsed_fields": parsed_fields})
    structured_rows.append({
        "file":               item["file"],
        "file_path":          str(item["path"]),
        "run_date":           RUN_DATE,
        "user_id":            item["user_id"],
        "transaction_date":   parsed_fields.get("transaction_date"),
        "description":        parsed_fields.get("description"),
        "transaction_code":   parsed_fields.get("transaction_code"),
        "transaction_amount": parsed_fields.get("transaction_amount"),
        "balance":            parsed_fields.get("balance"),
        "interest_rate":      parsed_fields.get("interest_rate"),
        "signature":          parsed_fields.get("signature"),
    })

print("=== STEP 4: PARSE & STRUCTURE ===")
df_structured = pd.DataFrame(structured_rows)
display(df_structured)

## Step 5 – Confidence & Save
Tính confidence theo OCR score trung bình và tỉ lệ field plausible; merge vào kết quả cuối.

In [ ]:
# Step 5: Confidence Scoring
confidence_rows = []

for item in structured_docs:
    scores = [float(p["score"]) for p in item["field_raw"].values() if p.get("score") is not None]
    plaus  = sum(1 for fn, pf in item["parsed_fields"].items() if is_plausible(fn, pf))
    total  = len(item["field_raw"])
    ocr_conf   = float(np.mean(scores)) if scores else 0.0
    parse_conf = plaus / max(1, total)
    final_conf = round(0.70 * ocr_conf + 0.30 * parse_conf, 4)
    confidence_rows.append({
        "file":             item["file"],
        "user_id":          item["user_id"],
        "ocr_confidence":   round(ocr_conf, 4),
        "parse_confidence": round(parse_conf, 4),
        "plausible_fields": plaus,
        "total_fields":     total,
        "final_confidence": final_conf,
    })

print("=== STEP 5: CONFIDENCE ===")
df_confidence = pd.DataFrame(confidence_rows)
display(df_confidence)

df_final = df_structured.merge(
    df_confidence[["file", "user_id", "final_confidence", "ocr_confidence", "parse_confidence", "plausible_fields"]],
    on=["file", "user_id"], how="left",
)
print("=== FINAL RESULT ===")
display(df_final)

In [ ]:
# Save extraction result
OUT_CSV = PROJECT_ROOT / "data" / "unstructured" / "extracted" / f"savings_book_roi_extractions_{RUN_DATE}.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_final.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved: {OUT_CSV}")